# §4 Individual with ML#2 filter (v12 top-50, V4 filter)

Per-combo metrics and equity/drawdown curves after applying the
ML#2 booster + pooled-R:R isotonic calibrator filter.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation._top_perf_common import (
    STARTING_EQUITY, POLICIES,
    apply_sizing, metrics_from_pnl, monte_carlo,
    load_setup,
    plot_indiv_equity, plot_indiv_dd,
    plot_combined_equity, plot_combined_dd,
    plot_ml2_indiv_equity, plot_ml2_indiv_dd,
    plot_ml2_combined_equity, plot_ml2_combined_dd,
    plot_mc_sims, plot_mc_pnl, plot_mc_sharpe, plot_mc_dd,
)

_ctx = load_setup(cost_per_contract_rt=0.0, top_strategies_path=REPO / 'evaluation' / 'top_strategies_v12_top50.json', version='v4')
bars            = _ctx['bars']
YEARS_SPAN      = _ctx['years_span']
strategies      = _ctx['strategies']
results_raw     = _ctx['results_raw']
combined_raw    = _ctx['combined_raw']
combos_ml2      = _ctx['combos_ml2']
s4_pnl_by_combo = _ctx['s4_pnl_by_combo']
ml2_portfolio   = _ctx['ml2_portfolio']


Top-K source: top_strategies_v12_top50.json


Test partition: 514,563 bars  2024-10-22 05:08:00 -> 2026-04-08 20:20:00
Years span: 1.461  (used to annualize Sharpe)
Loaded 50 strategies.
Loaded results_raw from cache (50 combos).
Combined unfiltered trades: 24,690
Loaded combos_ml2 from cache (50 combos).
ML2 portfolio trade counts: {'fixed_dollars_500': 681}


In [2]:
rows = []
for cid, entry in s4_pnl_by_combo.items():
    pnl_base = entry['pnl_base']; risk_base = entry['risk_base']
    if len(pnl_base) == 0:
        for policy in POLICIES:
            rows.append({'combo_id': cid, 'policy': policy,
                         **metrics_from_pnl(np.array([]), YEARS_SPAN, policy=policy)})
        continue
    r_mult = np.where(risk_base > 0, pnl_base / risk_base, 0.0)
    for policy in POLICIES:
        pnl = entry['by_policy'][policy]
        rows.append({'combo_id': cid, 'policy': policy,
                     **metrics_from_pnl(pnl, YEARS_SPAN, policy=policy, r=r_mult)})
perf4 = pd.DataFrame(rows)
perf4

,combo_id,policy,n_trades,trades_per_year,win_rate,total_pnl_dollars,total_return_pct,sharpe_ratio,max_drawdown_pct,max_drawdown_dollars
0,v11_2646,fixed_dollars_500,8,5.5,0.2500,1299.88,2.60,0.6615,0.80,399.79
1,v11_391,fixed_dollars_500,3,2.1,0.3333,473.11,0.95,0.2408,1.54,768.21
2,v11_28651,fixed_dollars_500,8,5.5,0.1250,-1255.23,-2.51,-0.4045,5.73,2866.83
3,v11_17782,fixed_dollars_500,6,4.1,0.3333,350.44,0.70,0.1521,2.58,1331.26
4,v11_263,fixed_dollars_500,57,39.0,0.3860,1092.15,2.18,0.1968,6.11,3260.57
5,v11_18020,fixed_dollars_500,9,6.2,0.2222,-19.61,-0.04,-0.0080,3.39,1710.06
6,v11_12101,fixed_dollars_500,4,2.7,0.2500,295.01,0.59,0.1443,2.09,1047.30
7,v11_27291,fixed_dollars_500,12,8.2,0.3333,-1246.96,-2.49,-0.5720,3.07,1536.25
8,v11_3547,fixed_dollars_500,3,2.1,0.0000,-974.48,-1.95,-1.6546,1.95,974.48
9,v11_25420,fixed_dollars_500,27,18.5,0.1481,-3697.40,-7.39,-0.9804,11.94,6113.72


In [3]:
plot_ml2_indiv_equity(s4_pnl_by_combo, bars, 'fixed_dollars_500')

In [4]:
plot_ml2_indiv_dd(s4_pnl_by_combo, bars, 'fixed_dollars_500')